# Surrogate v2 — Plan + Knobs + Workload → Latency / Ranking (Improved)

Improvements over v1:
- **Phase 1**: `db_type` flag added from the start (fixes transfer dimension bug). Plan parse-failure flag added. Near-constant feature pruning.
- **Phase 2**: Log-transform target. Full GroupKFold CV with early stopping. Feature importance diagnostics.
- **Phase 3**: LambdaRank trained on ALL PG data with tuned hyperparams.
- **Phase 4**: Held-out MySQL test split. Pseudo-label quality gate. Importance-weighted PG rows. Two-stage fine-tuning. Includes experimental Residual Learning and Weighted Joint Training for regression transfer.
- **Cross-cutting**: Artifact versioning with config hash. Reproducibility guards.

Phases:
- Phase 1: DB-agnostic representations (plan, knobs, workload) + domain flag
- Phase 2: PostgreSQL regression baseline — full CV, log-target, early stopping, diagnostics
- Phase 3: LambdaRank on PostgreSQL
- Phase 4: Transfer to MySQL — held-out split, pseudo-label gate, two-stage fine-tune, importance weighting
- Phase 5: Validate + save versioned artifacts

Uses repo CSVs:
- `surrogate/cost_model_collected.csv`
- `surrogate/cost_model_run_history.csv`

In [10]:
# If you don't have LightGBM installed in your env, uncomment:
# !pip install lightgbm joblib scikit-learn pandas numpy

from __future__ import annotations

import ast
import hashlib
import json
import math
import os
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

# ── Reproducibility ──────────────────────────────────────────────────────────
# LightGBM ignores np.random.seed; set PYTHONHASHSEED for full reproducibility.
os.environ.setdefault('PYTHONHASHSEED', '7')
RANDOM_SEED = 7
np.random.seed(RANDOM_SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
def find_repo_root(start: Path) -> Path:
    start = start.resolve()

    # Case A: running from repo root (expects surrogate/...) or any subdir.
    for p in [start, *start.parents]:
        if (p / 'surrogate' / 'cost_model_collected.csv').exists() and (
            p / 'surrogate' / 'cost_model_run_history.csv'
        ).exists():
            return p

    # Case B: running from inside the surrogate dir.
    for p in [start, *start.parents]:
        if p.name == 'surrogate' and (p / 'cost_model_collected.csv').exists() and (
            p / 'cost_model_run_history.csv'
        ).exists():
            return p.parent

    raise FileNotFoundError(
        'Could not locate repo root containing surrogate/cost_model_collected.csv and surrogate/cost_model_run_history.csv. '
        f'started from: {start}'
    )


ROOT = find_repo_root(Path.cwd())
COLLECTED_CSV = ROOT / 'surrogate' / 'cost_model_collected.csv'
RUN_HISTORY_CSV = ROOT / 'surrogate' / 'cost_model_run_history.csv'

# ── Config ───────────────────────────────────────────────────────────────────
PLAN_REPR            = os.environ.get('PLAN_REPR', 'structured')  # 'embedding' | 'structured'
REL_LEVELS           = 5      # relevance label granularity for LambdaRank
N_CV_FOLDS           = 5      # GroupKFold folds for PG baseline
PSEUDO_SPEARMAN_GATE = 0.0    # skip pseudo-labels for workloads with corr below this
PSEUDO_WEIGHT        = 0.3    # weight for pseudo-labelled MySQL rows

# ── Artifact versioning ──────────────────────────────────────────────────────
# Hash of key config so runs never silently overwrite each other.
_cfg_hash    = hashlib.md5(f'{PLAN_REPR}-{REL_LEVELS}-{RANDOM_SEED}'.encode()).hexdigest()[:8]
ARTIFACT_DIR = ROOT / 'surrogate' / 'artifacts' / 'transfer_rank_surrogate' / _cfg_hash
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print('cwd         =', Path.cwd().resolve())
print('repo root   =', ROOT)
print('PLAN_REPR   =', PLAN_REPR)
print('cfg_hash    =', _cfg_hash)
print('Collected   =', COLLECTED_CSV)
print('Run history =', RUN_HISTORY_CSV)
print('Artifact dir=', ARTIFACT_DIR)


cwd         = /home/E2ETune-AI4DB/surrogate
repo root   = /home/E2ETune-AI4DB
PLAN_REPR   = structured
cfg_hash    = 6ecfae00
Collected   = /home/E2ETune-AI4DB/surrogate/cost_model_collected.csv
Run history = /home/E2ETune-AI4DB/surrogate/cost_model_run_history.csv
Artifact dir= /home/E2ETune-AI4DB/surrogate/artifacts/transfer_rank_surrogate/6ecfae00


In [11]:
# ── Parsing helpers ──────────────────────────────────────────────────────────

def safe_parse_json_list(value: Any) -> Optional[list]:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    if isinstance(value, list):
        return value
    if not isinstance(value, str):
        return None
    s = value.strip()
    if not s:
        return None
    try:
        parsed = json.loads(s)
        return parsed if isinstance(parsed, list) else None
    except Exception:
        pass
    try:
        parsed = ast.literal_eval(s)
        return parsed if isinstance(parsed, list) else None
    except Exception:
        return None


def safe_parse_emb_vector(value: Any) -> Optional[np.ndarray]:
    lst = safe_parse_json_list(value)
    if lst is None:
        return None
    try:
        arr = np.asarray(lst, dtype=float)
        if arr.ndim != 1 or arr.size == 0:
            return None
        if not np.isfinite(arr).all():
            return None
        return arr
    except Exception:
        return None


# ── Load & merge data ────────────────────────────────────────────────────────
collected = pd.read_csv(COLLECTED_CSV, low_memory=False)
run_hist  = pd.read_csv(RUN_HISTORY_CSV, low_memory=False)

KEY_COL    = 'metadata.workload_key'
TARGET_COL = 'target.cost'

print('collected:', collected.shape)
print('run_hist :', run_hist.shape)

if KEY_COL not in run_hist.columns:
    raise KeyError(f"Missing {KEY_COL} in run history")
if KEY_COL not in collected.columns:
    raise KeyError(f"Missing {KEY_COL} in collected")

common_keys = [
    c for c in [
        'metadata.benchmark', 'metadata.db_engine', 'metadata.hardware',
        'metadata.hardware_specs.cores', 'metadata.hardware_specs.ram_gb',
        'metadata.hardware_specs.threads', 'metadata.workload', KEY_COL,
    ]
    if c in run_hist.columns and c in collected.columns
]

keep_cols = [
    c for c in collected.columns
    if c.startswith('collected.') or c.startswith('metadata.') or c == 'qp_emb_vector'
]

merged = run_hist.merge(collected[keep_cols], on=common_keys, how='left')

if TARGET_COL not in merged.columns:
    raise KeyError(f'Missing {TARGET_COL} after merge')

# ── Target preprocessing ─────────────────────────────────────────────────────
# `target.cost` mixes:
# - OLAP workloads: latency (positive)
# - OLTP workloads: negative throughput (negative)
# Convert to a single positive "latency-like" target where lower is better:
# - latency workloads: latency_like = cost (require cost > 0)
# - throughput workloads: latency_like = 1 / (-cost) (require cost < 0)

raw_cost = pd.to_numeric(merged[TARGET_COL], errors='coerce')
raw_cost = raw_cost.replace([np.inf, -np.inf], np.nan)

wk_series = merged[KEY_COL].astype(str)

wk_cost = pd.DataFrame({'wk': wk_series, 'c': raw_cost})
wk_stats = wk_cost.groupby('wk')['c'].agg(
    n='count',
    neg_frac=lambda s: float((s < 0).mean()),
    pos_frac=lambda s: float((s > 0).mean()),
    median='median',
)

wk_is_throughput = (wk_stats['median'] < 0)
is_throughput = wk_series.map(wk_is_throughput.to_dict()).fillna(False).to_numpy(dtype=bool)

raw_arr = raw_cost.to_numpy(dtype=float)
latency_like = np.full(len(raw_arr), np.nan, dtype=float)

lat_mask = ~is_throughput
lat_vals = raw_arr[lat_mask]
latency_like[lat_mask] = np.where(lat_vals > 0, lat_vals, np.nan)

thr_mask = is_throughput
thr = -raw_arr[thr_mask]
thr = np.where(thr > 0, thr, np.nan)
latency_like[thr_mask] = 1.0 / thr

latency_like = np.where(np.isfinite(latency_like), latency_like, np.nan)
keep_target = np.isfinite(latency_like)

dropped = int((~keep_target).sum())
if dropped:
    print(f'Dropping {dropped} rows with invalid {TARGET_COL} for latency-like target')

merged = merged.loc[keep_target].reset_index(drop=True)
y  = latency_like[keep_target].astype(float)
wk = merged[KEY_COL].astype(str).to_numpy()

print('objective type counts (workloads):', {
    'throughput_wk': int(wk_is_throughput.sum()),
    'latency_wk': int((~wk_is_throughput).sum()),
})
print('objective type counts (rows):', {
    'throughput_rows': int(thr_mask.sum()),
    'latency_rows': int(lat_mask.sum()),
})

print('merged:', merged.shape)
print('y(latency_like) stats - min:', float(np.min(y)), ' mean:', float(np.mean(y)), ' max:', float(np.max(y)))
print('unique workload keys:', len(np.unique(wk)))


collected: (264, 109)
run_hist : (22176, 81)
Dropping 104 rows with invalid target.cost for latency-like target
objective type counts (workloads): {'throughput_wk': 109, 'latency_wk': 154}
objective type counts (rows): {'throughput_rows': 7935, 'latency_rows': 14241}
merged: (22072, 182)
y(latency_like) stats - min: 2.1742046965080753e-05  mean: 2.1493904833769646  max: 112.7186926606944
unique workload keys: 263


In [12]:
# Phase 1.1 - Query plan encoding (DB-agnostic structured features)
#
# IMPROVEMENT: added `plan.parse_failed` flag so the model can learn when
# plan signal is missing rather than silently receiving all-zeros.

OP_ALIASES = {
    'seq scan': 'table_scan', 'table scan': 'table_scan',
    'index scan': 'index_scan', 'index only scan': 'index_scan',
    'hash join': 'join', 'merge join': 'join', 'nested loop': 'join', 'join': 'join',
    'sort': 'sort', 'aggregate': 'aggregate',
    'limit': 'limit', 'gather': 'gather', 'gather merge': 'gather',
}


def normalize_op(op: str) -> str:
    s = re.sub(r'\s+', ' ', (op or '').strip().lower())
    return OP_ALIASES.get(s, s)


def structured_plan_features(plan_list: List[str]) -> Dict[str, float]:
    feats: Dict[str, float] = {}
    if not plan_list:
        feats['plan.parse_failed'] = 1.0
        return feats

    ops: List[str] = []
    costs: List[float] = []
    rows: List[float] = []
    any_index = join_hash = join_merge = join_nested = 0

    for p in plan_list:
        if not isinstance(p, str) or not p.strip():
            continue
        # Match operation names like "Hash Join(" or "Seq Scan("
        for raw in re.findall(r'([A-Za-z][A-Za-z ]+?)\(', p):
            norm = normalize_op(raw)
            ops.append(norm)
            rl = raw.lower()
            join_hash   += int('hash join'   in rl)
            join_merge  += int('merge join'  in rl)
            join_nested += int('nested loop' in rl)
            any_index    = any_index or int('index' in rl)
        for m in re.finditer(r'cost=([0-9]+(?:\.[0-9]+)?)', p):
            costs.append(float(m.group(1)))
        for m in re.finditer(r'rows=([0-9]+(?:\.[0-9]+)?)', p):
            rows.append(float(m.group(1)))

    if not ops:
        feats['plan.parse_failed'] = 1.0
        return feats

    total = float(len(ops))
    feats['plan.parse_failed']     = 0.0
    feats['plan.num_plans']        = float(len(plan_list))
    feats['plan.total_nodes']      = total
    feats['plan.any_index']        = float(any_index)
    feats['plan.join_hash_prop']   = float(join_hash)   / total
    feats['plan.join_merge_prop']  = float(join_merge)  / total
    feats['plan.join_nested_prop'] = float(join_nested) / total
    feats['plan.avg_cost']         = float(np.mean(costs)) if costs else 0.0
    feats['plan.max_cost']         = float(np.max(costs))  if costs else 0.0
    feats['plan.avg_rows']         = float(np.mean(rows))  if rows  else 0.0
    feats['plan.max_rows']         = float(np.max(rows))   if rows  else 0.0

    counts: Dict[str, int] = {}
    for op in ops:
        counts[op] = counts.get(op, 0) + 1
    for op, cnt in counts.items():
        feats[f'plan.op_prop.{op}'] = float(cnt) / total

    return feats


plan_src = collected[[KEY_COL, 'collected.query_plans']].copy()
plan_src['__plans'] = plan_src['collected.query_plans'].apply(safe_parse_json_list)

feat_rows = []
for wk_i, plans in zip(plan_src[KEY_COL].astype(str), plan_src['__plans']):
    f = structured_plan_features(plans or [])
    f[KEY_COL] = wk_i
    feat_rows.append(f)

plan_struct_df = pd.DataFrame(feat_rows).fillna(0.0)
merged_struct  = merged[[KEY_COL]].astype(str).merge(plan_struct_df, on=KEY_COL, how='left').fillna(0.0)

plan_struct_cols   = [c for c in merged_struct.columns if c != KEY_COL]
plan_struct_matrix = merged_struct[plan_struct_cols].to_numpy(dtype=float)

parse_fail_rate = plan_struct_df.get('plan.parse_failed', pd.Series([0])).mean()
print('structured plan dims:', plan_struct_matrix.shape)
print('parse failure rate  :', f'{parse_fail_rate:.1%}')
print('example plan cols   :', plan_struct_cols[:12])


structured plan dims: (22072, 63)
parse failure rate  : 14.0%
example plan cols   : ['plan.parse_failed', 'plan.num_plans', 'plan.total_nodes', 'plan.any_index', 'plan.join_hash_prop', 'plan.join_merge_prop', 'plan.join_nested_prop', 'plan.avg_cost', 'plan.max_cost', 'plan.avg_rows', 'plan.max_rows', 'plan.op_prop.aggregate']


In [13]:
# Phase 1.2 - Knob encoding  |  Phase 1.3 - Workload encoding
#
# IMPROVEMENT: drop near-constant columns before any model sees them.
# Constant features add noise and inflate the importance of real signals.

workload_cols_raw = [c for c in merged.columns if c.startswith('collected.workload_features.')]
knob_cols_raw     = [c for c in merged.columns if c.startswith('features.')]

LOG_HINTS         = ('buffer', 'mem', 'cache', 'size', 'capacity', 'tmp', 'wal', 'shared', 'work_mem')
log_knob_cols_raw = [c for c in knob_cols_raw if any(h in c.lower() for h in LOG_HINTS)]


def drop_near_constant(df: pd.DataFrame, cols: List[str], threshold: float = 0.99) -> List[str]:
    """Drop columns where >= threshold fraction of rows share the modal value."""
    keep = []
    for c in cols:
        arr = pd.to_numeric(df[c], errors='coerce').fillna(0)
        mode_frac = arr.value_counts(normalize=True).iloc[0] if len(arr) > 0 else 1.0
        if mode_frac < threshold:
            keep.append(c)
    dropped = set(cols) - set(keep)
    if dropped:
        print(f'  Dropped {len(dropped)} near-constant cols:', sorted(dropped)[:5],
              '...' if len(dropped) > 5 else '')
    return keep


print('Pruning near-constant knob columns...')
knob_cols     = drop_near_constant(merged, knob_cols_raw)
log_knob_cols = [c for c in log_knob_cols_raw if c in knob_cols]

print('Pruning near-constant workload columns...')
workload_cols = drop_near_constant(merged, workload_cols_raw)


def encode_knobs(
    df: pd.DataFrame,
    cols: List[str],
    log_cols: List[str],
) -> Tuple[np.ndarray, Dict[str, Dict[str, float]]]:
    Xk = df[cols].copy()
    for c in cols:
        Xk[c] = pd.to_numeric(Xk[c], errors='coerce')
    for c in log_cols:
        Xk[c] = np.log1p(Xk[c].where(Xk[c] >= 0))
    stats: Dict[str, Dict[str, float]] = {}
    for c in cols:
        arr   = Xk[c].to_numpy(dtype=float)
        mn    = float(np.nanmin(arr)) if np.isfinite(np.nanmin(arr)) else 0.0
        mx    = float(np.nanmax(arr)) if np.isfinite(np.nanmax(arr)) else mn
        denom = (mx - mn) if (mx - mn) != 0 else 1.0
        Xk[c] = ((Xk[c] - mn) / denom).clip(0.0, 1.0)
        stats[c] = {'min': mn, 'max': mx}
    return Xk.fillna(0.0).to_numpy(dtype=float), stats


knob_matrix,  knob_stats = encode_knobs(merged, knob_cols, log_knob_cols)
workload_matrix           = merged[workload_cols].fillna(0.0).to_numpy(dtype=float)

print('knob dims    :', knob_matrix.shape)
print('workload dims:', workload_matrix.shape)
print('log knobs    :', len(log_knob_cols))


Pruning near-constant knob columns...
Pruning near-constant workload columns...
  Dropped 5 near-constant cols: ['collected.workload_features.operator_proportions.max_agg', 'collected.workload_features.operator_proportions.min_agg', 'collected.workload_features.table_access_frequency.customer_summary', 'collected.workload_features.table_access_frequency.movie_info_idx', 'collected.workload_features.table_access_frequency.web_returns'] 
knob dims    : (22072, 71)
workload dims: (22072, 79)
log knobs    : 21


In [14]:
# Phase 1.4 - Final input: X = [plan_features, knobs, workload, db_type]
#
# IMPROVEMENT: `db_type` flag (0=PostgreSQL, 1=MySQL) is added HERE, BEFORE
# any model is trained. In v1 it was added only during MySQL transfer, causing
# a silent feature-dimension mismatch when warm-starting the ranker.

def get_plan_matrix(df: pd.DataFrame) -> Tuple[np.ndarray, Dict[str, object]]:
    if PLAN_REPR == 'structured':
        return plan_struct_matrix, {'plan_repr': 'structured', 'cols': plan_struct_cols}

    if 'qp_emb_vector' not in df.columns:
        raise KeyError(
            'Missing qp_emb_vector; run surrogate/add_query_plan_embeddings.py '
            'or set PLAN_REPR=structured'
        )
    vecs = df['qp_emb_vector'].apply(safe_parse_emb_vector).tolist()
    dims = [v.size for v in vecs if v is not None]
    if not dims:
        raise ValueError('No valid qp_emb_vector; set PLAN_REPR=structured or backfill.')

    dim = int(np.median(dims))
    mat = np.zeros((len(vecs), dim), dtype=float)
    missing = 0
    for i, v in enumerate(vecs):
        if v is None:
            missing += 1
            continue
        if v.size != dim:
            raise ValueError(f'Embedding dim mismatch at row {i}: {v.size} != {dim}')
        mat[i, :] = v

    return mat, {'plan_repr': 'embedding', 'dim': dim, 'missing': missing}


plan_matrix, plan_meta = get_plan_matrix(merged)

engine = (
    merged['metadata.db_engine'].astype(str).str.lower()
    if 'metadata.db_engine' in merged.columns
    else pd.Series(['postgresql'] * len(merged))
)
db_type_flag = (engine == 'mysql').astype(int).to_numpy().reshape(-1, 1)  # 0=PG, 1=MySQL

X = np.hstack([plan_matrix, knob_matrix, workload_matrix, db_type_flag]).astype(float)

FEATURE_NAMES = plan_struct_cols + knob_cols + workload_cols + ['db_type']

print('plan dims  :', plan_matrix.shape)
print('X dims     :', X.shape)
print('plan_meta  :', plan_meta)
print('PG rows    :', int((engine == 'postgresql').sum()))
print('MySQL rows :', int((engine == 'mysql').sum()))


plan dims  : (22072, 63)
X dims     : (22072, 214)
plan_meta  : {'plan_repr': 'structured', 'cols': ['plan.parse_failed', 'plan.num_plans', 'plan.total_nodes', 'plan.any_index', 'plan.join_hash_prop', 'plan.join_merge_prop', 'plan.join_nested_prop', 'plan.avg_cost', 'plan.max_cost', 'plan.avg_rows', 'plan.max_rows', 'plan.op_prop.aggregate', 'plan.op_prop.gather', 'plan.op_prop.table_scan', 'plan.op_prop.sort', 'plan.op_prop.join', 'plan.op_prop.hash', 'plan.op_prop.limit', 'plan.op_prop.index_scan', 'plan.op_prop.bitmap heap scan', 'plan.op_prop.bitmap index scan', 'plan.op_prop.materialize', 'plan.op_prop.cte scan', 'plan.op_prop.series', 'plan.op_prop.lower', 'plan.op_prop.upper', 'plan.op_prop.as g', 'plan.op_prop.any', 'plan.op_prop.subquery scan', 'plan.op_prop.unique', 'plan.op_prop.and', 'plan.op_prop.values', 'plan.op_prop.or', 'plan.op_prop.sv', 'plan.op_prop.onxl', 'plan.op_prop.info', 'plan.op_prop.queryblock', 'plan.op_prop.orderingoperation', 'plan.op_prop.groupingoperati

In [15]:
# Shared helpers

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error


def spearman_corr(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    s1 = pd.Series(y_true).rank(method='average')
    s2 = pd.Series(y_pred).rank(method='average')
    return float(s1.corr(s2))


def eval_per_workload(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    wk_keys: np.ndarray,
) -> Dict[str, float]:
    df = pd.DataFrame({'wk': wk_keys, 'y': y_true, 'p': y_pred})
    spears, top1, top3 = [], [], []
    for _, g in df.groupby('wk'):
        if len(g) < 2:
            continue
        spears.append(spearman_corr(g['y'].to_numpy(), g['p'].to_numpy()))
        best_true = int(g['y'].idxmin())
        top1_pred = int(g['p'].idxmin())
        top3_pred = g.nsmallest(min(3, len(g)), 'p').index.tolist()
        top1.append(1.0 if top1_pred == best_true else 0.0)
        top3.append(1.0 if best_true in top3_pred else 0.0)
    return {
        'spearman_mean': float(np.nanmean(spears)) if spears else float('nan'),
        'top1_acc'     : float(np.mean(top1))      if top1   else float('nan'),
        'top3_recall'  : float(np.mean(top3))      if top3   else float('nan'),
        'n_workloads'  : len(spears),
    }


def group_sizes(sorted_wk: np.ndarray) -> List[int]:
    sizes, last, cnt = [], None, 0
    for k in sorted_wk:
        if last is None:
            last, cnt = k, 1
        elif k == last:
            cnt += 1
        else:
            sizes.append(cnt)
            last, cnt = k, 1
    if last is not None:
        sizes.append(cnt)
    return sizes


def relevance_labels(
    lat: np.ndarray,
    wk_keys: np.ndarray,
    levels: int = REL_LEVELS,
) -> np.ndarray:
    """Lower latency -> higher relevance label (levels-1 = best)."""
    rel = np.zeros_like(lat, dtype=int)
    df  = pd.DataFrame({'wk': wk_keys, 'y': lat})
    for _, idx in df.groupby('wk').groups.items():
        y_g   = df.loc[idx, 'y']
        ranks = (y_g.rank(method='first', ascending=True).to_numpy() - 1)
        if len(ranks) == 1:
            rel[idx] = levels - 1
            continue
        q        = ranks / max(1, (len(ranks) - 1))
        rel[idx] = np.floor((1.0 - q) * (levels - 1) + 1e-9).astype(int)
    return rel


print('Helpers loaded.')


Helpers loaded.


In [16]:
# Phase 2 - Regression baseline on PostgreSQL
#
# IMPROVEMENTS vs v1:
#   1. Log1p-transform target - right-skewed latency distributions hurt RMSE.
#      Invert with expm1 at eval time so RMSE stays interpretable.
#   2. Full GroupKFold CV (all N_CV_FOLDS folds), not just next(iter(...)).
#   3. Early stopping on an inner val split within each fold.
#   4. Feature importance diagnostics: checks plan vs knob vs workload split.
#   5. Final regressor retrained on ALL PG data (used for pseudo-label quality later).

try:
    from lightgbm import LGBMRegressor, early_stopping, log_evaluation
    _HAS_LGBM = True
except Exception:
    from sklearn.ensemble import HistGradientBoostingRegressor
    _HAS_LGBM = False
    print('WARNING: LightGBM not found - falling back to HistGradientBoostingRegressor (no early stopping).')

is_pg  = (engine == 'postgresql').to_numpy()
X_pg   = X[is_pg]
y_pg   = y[is_pg]
wk_pg  = wk[is_pg]

# IMPROVEMENT 1: log-transform the target
y_pg_log = np.log1p(y_pg)

n_splits = min(N_CV_FOLDS, len(np.unique(wk_pg)))
gkf      = GroupKFold(n_splits=n_splits)

cv_rmse, cv_spearman, fi_accum = [], [], None

for fold, (tr_idx, te_idx) in enumerate(gkf.split(X_pg, y_pg_log, groups=wk_pg)):
    X_tr, X_te = X_pg[tr_idx], X_pg[te_idx]
    y_tr, y_te = y_pg_log[tr_idx], y_pg_log[te_idx]
    wk_te      = wk_pg[te_idx]

    # Inner val split for early stopping (last 15% of training workload groups)
    tr_groups    = np.unique(wk_pg[tr_idx])
    n_val_groups = max(1, int(0.15 * len(tr_groups)))
    val_groups   = set(tr_groups[-n_val_groups:])
    val_mask     = np.array([g in val_groups for g in wk_pg[tr_idx]])

    X_tr2, X_val = X_tr[~val_mask], X_tr[val_mask]
    y_tr2, y_val = y_tr[~val_mask], y_tr[val_mask]

    if _HAS_LGBM:
        reg = LGBMRegressor(
            n_estimators     = 3000,
            learning_rate    = 0.03,
            num_leaves       = 63,
            min_child_samples= 20,
            lambda_l1        = 0.1,
            lambda_l2        = 0.1,
            subsample        = 0.9,
            colsample_bytree = 0.9,
            random_state     = RANDOM_SEED,
            n_jobs           = -1,
        )
        reg.fit(
            X_tr2, y_tr2,
            eval_set=[(X_val, y_val)],
            callbacks=[early_stopping(50, verbose=False), log_evaluation(0)],
        )
    else:
        reg = HistGradientBoostingRegressor(random_state=RANDOM_SEED)
        reg.fit(X_tr2, y_tr2)

    pred_log = reg.predict(X_te)
    pred     = np.expm1(pred_log)   # invert log1p for interpretable RMSE
    y_te_raw = np.expm1(y_te)

    rmse_fold = math.sqrt(mean_squared_error(y_te_raw, pred))
    sp_fold   = spearman_corr(y_te_raw, pred)
    cv_rmse.append(rmse_fold)
    cv_spearman.append(sp_fold)

    if _HAS_LGBM and hasattr(reg, 'feature_importances_'):
        fi = reg.feature_importances_
        fi_accum = fi if fi_accum is None else fi_accum + fi

    best_it = getattr(reg, 'best_iteration_', '-') if _HAS_LGBM else '-'
    print(f'  Fold {fold+1}/{n_splits}  RMSE: {rmse_fold:.4f}  Spearman: {sp_fold:.4f}  best_iter: {best_it}')

print(f'\nCV RMSE     : {np.mean(cv_rmse):.4f} +/- {np.std(cv_rmse):.4f}')
print(f'CV Spearman : {np.mean(cv_spearman):.4f} +/- {np.std(cv_spearman):.4f}')

# IMPROVEMENT 4: Feature importance diagnostics
if fi_accum is not None and len(FEATURE_NAMES) == len(fi_accum):
    fi_df    = pd.DataFrame({'feature': FEATURE_NAMES, 'importance': fi_accum})
    fi_df    = fi_df.sort_values('importance', ascending=False)
    total_fi = fi_df['importance'].sum() or 1.0

    plan_fi = fi_df[fi_df['feature'].str.startswith('plan.')]['importance'].sum()
    knob_fi = fi_df[fi_df['feature'].str.startswith('features.')]['importance'].sum()
    wkld_fi = fi_df[fi_df['feature'].str.startswith('collected.workload')]['importance'].sum()
    db_fi   = fi_df[fi_df['feature'] == 'db_type']['importance'].sum()

    print(f'\nFeature group importance (summed across folds):')
    print(f'  Plan features    : {plan_fi/total_fi:.1%}')
    print(f'  Knob features    : {knob_fi/total_fi:.1%}')
    print(f'  Workload features: {wkld_fi/total_fi:.1%}')
    print(f'  db_type flag     : {db_fi/total_fi:.1%}')
    print(f'\nTop-15 features:')
    print(fi_df.head(15).to_string(index=False))

    if plan_fi / total_fi < 0.05:
        print('\nWARNING: Plan features < 5% of total importance.')
        print('  Check plan parsing or try PLAN_REPR=embedding.')

# IMPROVEMENT 5: Retrain final PG regressor on ALL PG data
if _HAS_LGBM:
    best_n    = getattr(reg, 'best_iteration_', 1000)
    reg_final = LGBMRegressor(
        n_estimators=best_n, learning_rate=0.03, num_leaves=63,
        min_child_samples=20, lambda_l1=0.1, lambda_l2=0.1,
        subsample=0.9, colsample_bytree=0.9,
        random_state=RANDOM_SEED, n_jobs=-1,
    )
else:
    reg_final = HistGradientBoostingRegressor(random_state=RANDOM_SEED)

reg_final.fit(X_pg, y_pg_log)
print('\nFinal PG regressor trained on all PG data.')


[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.355279 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 11830
[LightGBM] [Info] Number of data points in the train set: 14637, number of used features: 152
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Info] Start training from score 0.602691


/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
  Fold 1/5  RMSE: 0.8831  Spearman: 0.9308  best_iter: 363
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004272 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11839
[LightGBM] [Info] Number of data points in the train set: 14452, number of used features: 151
[LightGBM] [Warning] lambda_l

/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


...

[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
  Fold 3/5  RMSE: 1.0320  Spearman: 0.9535  best_iter: 268
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004461 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11809
[LightGBM] [Info] Number of data points in the train set: 14468, number of used features: 151
[LightGBM] [Warning] lambda_l

/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
  Fold 4/5  RMSE: 1.3970  Spearman: 0.9235  best_iter: 261
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004435 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11865
[LightGBM] [Info] Number of data points in the train set: 14414, number of used features: 152
[LightGBM] [Warning] lambda_l

/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
  Fold 5/5  RMSE: 0.8333  Spearman: 0.9653  best_iter: 239

CV RMSE     : 0.9991 +/- 0.2109
CV Spearman : 0.9417 +/- 0.0154

Feature group importance (summed across folds):
  Plan features    : 45.3%
  Knob features    : 43.4%
  Workload features: 11.3%
  db_type flag     : 0.0%

Top-15 features:
                                                   feature  importance
                                   features.shared_buffers       10455
                                             plan.avg_cost        6785
                                             plan.max_cost        6516
                                         features.work_mem        4834
                                            plan.num_plans        4361
                                       plan.join_hash_prop        391

In [17]:
# Phase 3 - LambdaRank on PostgreSQL
#
# Trained on ALL PG data. Tuned vs v1: min_child_samples, lambda_l1/l2,
# ndcg_eval_at=[1,3,5] for richer diagnostic output.

try:
    from lightgbm import LGBMRanker
except Exception as e:
    raise RuntimeError('LightGBM is required for LambdaRank. pip install lightgbm') from e

order_pg  = np.argsort(wk_pg)
X_pg_s    = X_pg[order_pg]
y_pg_s    = y_pg[order_pg]
wk_pg_s   = wk_pg[order_pg]

rel_pg    = relevance_labels(y_pg_s, wk_pg_s)
groups_pg = group_sizes(wk_pg_s)

ranker = LGBMRanker(
    objective        = 'lambdarank',
    metric           = 'ndcg',
    ndcg_eval_at     = [1, 3, 5],
    n_estimators     = 2000,
    learning_rate    = 0.03,
    num_leaves       = 63,
    min_child_samples= 20,
    lambda_l1        = 0.1,
    lambda_l2        = 0.1,
    subsample        = 0.9,
    colsample_bytree = 0.9,
    random_state     = RANDOM_SEED,
    n_jobs           = -1,
)
ranker.fit(X_pg_s, rel_pg, group=groups_pg)

print('PG ranker trained.  n_estimators =', ranker.n_estimators_)


[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1


/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006395 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12364
[LightGBM] [Info] Number of data points in the train set: 20758, number of used features: 155
PG ranker trained.  n_estimators = 2000


In [ ]:
# Phase 4 - Transfer to MySQL
#
# IMPROVEMENTS vs v1:
#   1. Held-out MySQL test split (20% of workload groups) - never seen during fine-tuning.
#   2. Pseudo-label quality gate - skip workloads where PG ranker Spearman < PSEUDO_SPEARMAN_GATE.
#   3. Importance weighting - weight PG rows by L2 similarity to MySQL distribution.
#   4. Two-stage fine-tuning:
#      Stage 1: PG (importance-weighted) + gated pseudo-labels at low LR.
#      Stage 2: Real MySQL labels only at even lower LR.
#   5. Zero-shot baseline reported alongside transfer results for fair comparison.

is_my  = (engine == 'mysql').to_numpy()

if not np.any(is_my):
    print('No MySQL rows found; skipping transfer.')
    ranker_my = None
    per_wk_my = per_wk_pg_on_my = {}
    spear_vals, good_wks = [], set()
else:
    X_my  = X[is_my]
    y_my  = y[is_my]
    wk_my = wk[is_my]

    # 4.0  Held-out MySQL test split (by workload group)
    unique_my_wks = np.unique(wk_my)
    n_test_wks    = max(1, int(0.2 * len(unique_my_wks)))
    rng           = np.random.default_rng(RANDOM_SEED)
    test_wks      = set(rng.choice(unique_my_wks, size=n_test_wks, replace=False))
    finetune_mask = np.array([w not in test_wks for w in wk_my])
    test_mask     = ~finetune_mask

    X_my_ft,   y_my_ft,   wk_my_ft   = X_my[finetune_mask],  y_my[finetune_mask],  wk_my[finetune_mask]
    X_my_test, y_my_test, wk_my_test = X_my[test_mask],      y_my[test_mask],      wk_my[test_mask]

    print(f'MySQL fine-tune : {len(X_my_ft)} rows / {len(np.unique(wk_my_ft))} workloads')
    print(f'MySQL held-out  : {len(X_my_test)} rows / {len(np.unique(wk_my_test))} workloads')

    # 4.1  Pseudo-label quality gate
    pseudo_scores   = ranker.predict(X_my_ft)
    pseudo_pred_lat = -pseudo_scores  # higher score = better rank = lower predicted latency

    wk_pseudo_spear: Dict[str, float] = {}
    for wk_i in np.unique(wk_my_ft):
        mask_i = wk_my_ft == wk_i
        if mask_i.sum() < 2:
            wk_pseudo_spear[wk_i] = float('nan')
            continue
        wk_pseudo_spear[wk_i] = spearman_corr(y_my_ft[mask_i], pseudo_pred_lat[mask_i])

    spear_vals = [s for s in wk_pseudo_spear.values() if not np.isnan(s)]
    
    if not spear_vals:
        print('\nPseudo-label Spearman: no valid workloads (need >=2 samples per workload).')
        print('Skipping pseudo-label gating and Stage 1 warm-up; doing Stage 2 (real-label refinement) only.')
        good_wks = set()
        pseudo_gate = np.zeros(len(wk_my_ft), dtype=bool)
    else:
        good_wks    = {w for w, s in wk_pseudo_spear.items() if not np.isnan(s) and s >= PSEUDO_SPEARMAN_GATE}
        pseudo_gate = np.array([w in good_wks for w in wk_my_ft])

        print(f'\nPseudo-label Spearman: mean={np.mean(spear_vals):.3f}  '
              f'min={np.min(spear_vals):.3f}  max={np.max(spear_vals):.3f}')
        print(f'Workloads passing gate (>={PSEUDO_SPEARMAN_GATE}): {len(good_wks)}/{len(np.unique(wk_my_ft))}')

        if np.mean(spear_vals) < 0:
            print('WARNING: PG ranker is negatively correlated with MySQL latency on average.')
            print('  Consider more PG data, better plan normalisation, or lower PSEUDO_SPEARMAN_GATE.')

    # 4.2  Importance-weight PG rows by similarity to MySQL distribution
    my_mean  = X_my_ft.mean(axis=0)
    pg_dists = np.linalg.norm(X_pg - my_mean, axis=1)
    pg_sim   = np.exp(-pg_dists / (pg_dists.std() + 1e-8))
    pg_sim   = pg_sim / (pg_sim.sum() + 1e-8) * len(pg_sim)  # rescale to unit mean
    print(f'\nPG importance weights  mean: {pg_sim.mean():.3f}  std: {pg_sim.std():.3f}')

    # 4.3  Stage 1: PG (importance-weighted) + gated pseudo-labels
    print('\n-- Stage 1: pseudo-label warm-up --')
    rel_pg_all = relevance_labels(y_pg, wk_pg)
    X_s1_my    = X_my_ft[pseudo_gate]
    y_s1_my    = y_my_ft[pseudo_gate]
    wk_s1_my   = wk_my_ft[pseudo_gate]
    pseudo_rel = relevance_labels(y_s1_my, wk_s1_my)

    X_s1   = np.vstack([X_pg,      X_s1_my])
    rel_s1 = np.concatenate([rel_pg_all, pseudo_rel])
    wk_s1  = np.concatenate([wk_pg,     wk_s1_my])
    w_s1   = np.concatenate([pg_sim,     np.ones(len(wk_s1_my)) * PSEUDO_WEIGHT])

    order_s1  = np.argsort(wk_s1)
    groups_s1 = group_sizes(wk_s1[order_s1])

    ranker_s1 = LGBMRanker(
        objective='lambdarank', metric='ndcg', ndcg_eval_at=[1, 3, 5],
        n_estimators=600,         # shorter: pseudo labels are noisy
        learning_rate=0.02,       # lower LR for pseudo-label stage
        num_leaves=63, min_child_samples=20, lambda_l1=0.1, lambda_l2=0.1,
        subsample=0.9, colsample_bytree=0.9,
        random_state=RANDOM_SEED, n_jobs=-1,
    )
    ranker_s1.fit(
        X_s1[order_s1], rel_s1[order_s1],
        group=groups_s1,
        sample_weight=w_s1[order_s1],
        init_model=ranker.booster_,  # warm-start from PG ranker
    )
    print('  Stage 1 done.')

    # 4.4  Stage 2: real MySQL labels only at lower LR
    print('-- Stage 2: real-label refinement --')
    real_rel_my = relevance_labels(y_my_ft, wk_my_ft)
    order_s2    = np.argsort(wk_my_ft)
    groups_s2   = group_sizes(wk_my_ft[order_s2])

    ranker_my = LGBMRanker(
        objective='lambdarank', metric='ndcg', ndcg_eval_at=[1, 3, 5],
        n_estimators=300,         # short: small real dataset, prevent overfit
        learning_rate=0.01,       # even lower LR for final real-label pass
        num_leaves=31,            # smaller trees -> less overfit on few samples
        min_child_samples=10, lambda_l1=0.2, lambda_l2=0.2,
        subsample=0.9, colsample_bytree=0.9,
        random_state=RANDOM_SEED, n_jobs=-1,
    )
    ranker_my.fit(
        X_my_ft[order_s2], real_rel_my[order_s2],
        group=groups_s2,
        init_model=ranker_s1.booster_,  # warm-start from stage 1
    )
    print('  Stage 2 done.')

    # 4.5  Evaluate on held-out MySQL test set
    score_pg_on_my  = ranker.predict(X_my_test)
    per_wk_pg_on_my = eval_per_workload(y_my_test, -score_pg_on_my, wk_my_test)

    score_my_test = ranker_my.predict(X_my_test)
    per_wk_my     = eval_per_workload(y_my_test, -score_my_test, wk_my_test)

    print('\n-- Held-out MySQL evaluation (never seen during fine-tuning) --')
    print(f'  Zero-shot (PG ranker -> MySQL) : {per_wk_pg_on_my}')
    print(f'  After transfer (Stage 1 + 2)   : {per_wk_my}')

    # 4.6 Optional: Residual Learning Regressor Transfer (Experiment)
    if getattr(reg_final, 'booster_', None) is not None:
        print('\n-- Residual Learning Regressor Transfer to MySQL --')
        
        # 1. Predict PG base on MySQL train/test
        pred_pg_log_ft   = reg_final.predict(X_my_ft)
        pred_pg_log_test = reg_final.predict(X_my_test)
        
        # 2. Targets in log space
        y_my_log_ft   = np.log1p(y_my_ft)
        
        # 3. Calculate residual on fine-tune set
        residual_ft = y_my_log_ft - pred_pg_log_ft
        
        # Stabilize residual learning by clipping extreme spikes (Issue 4)
        residual_ft = np.clip(residual_ft, -3, 3)
        
        # 4. Train residual model
        res_model = LGBMRegressor(
        n_estimators=300,
        learning_rate=0.01,
        num_leaves=31,
        min_child_samples=10,
        reg_alpha=0.2,
        reg_lambda=0.2,
        random_state=RANDOM_SEED,
        n_jobs=-1
    )
        res_model.fit(X_my_ft, residual_ft)
        
        # 5. Predict & correct on test set
        pred_res_test = res_model.predict(X_my_test)
        
        # Blended correction: shrink the residual slightly to prevent overfitting (Optional Upgrade)
        alpha_res = 0.7  # 1.0 = full residual addition, <1.0 = damped residual addition
        final_pred_log = pred_pg_log_test + (alpha_res * pred_res_test)
        
        # Inverse log transform (clip to prevent overflow on wild predictions)
        final_pred     = np.expm1(np.clip(final_pred_log, -20, 20))
        
        # 6. Evaluate
        rmse_res = math.sqrt(mean_squared_error(y_my_test, final_pred))
        sp_res   = spearman_corr(y_my_test, final_pred)
        
        print(f'  Residual Regressor -> RMSE: {rmse_res:.4f} | Spearman: {sp_res:.4f}')

    # 4.7 Optional: Weighted Joint Training Regressor (Cross-Validation & Mean Alignment)
    print('\n-- Weighted Joint Training Regressor Transfer to MySQL (GroupKFold CV) --')
    
    n_splits_my = min(N_CV_FOLDS, len(np.unique(wk_my)))
    gkf_my = GroupKFold(n_splits=n_splits_my)
    
    cv_rmse_joint = []
    cv_sp_joint = []
    
    y_my_log = np.log1p(y_my)
    
    for fold, (tr_idx, te_idx) in enumerate(gkf_my.split(X_my, y_my_log, groups=wk_my)):
        X_my_tr, X_my_te = X_my[tr_idx], X_my[te_idx]
        y_my_log_tr, y_my_log_te = y_my_log[tr_idx], y_my_log[te_idx]
        y_my_te_raw = y_my[te_idx]
        
        # 1. Similarity-based weights for this fold
        my_mean_tr = X_my_tr.mean(axis=0)
        pg_dists_fold = np.linalg.norm(X_pg - my_mean_tr, axis=1)
        pg_sim_fold = np.exp(-pg_dists_fold / (pg_dists_fold.std() + 1e-8))
        pg_sim_fold = pg_sim_fold / (pg_sim_fold.sum() + 1e-8) * len(pg_sim_fold)
        
        # 2. Target Alignment (Domain Adaptation Upgrade)
        # Shift PG targets to have the same mean as MySQL train targets.
        # This stops the model from spending capacity learning a flat latency gap between engines.
        pg_mean_offset = y_pg_log.mean() - y_my_log_tr.mean()
        y_pg_log_aligned = y_pg_log - pg_mean_offset
        
        # 3. Merge datasets
        X_joint = np.vstack([X_pg, X_my_tr])
        y_joint = np.concatenate([y_pg_log_aligned, y_my_log_tr])
        
        # 4. Sample weights
        w_joint = np.concatenate([
            0.05 * pg_sim_fold,  # Dialed down to 0.05 so PG acts purely as a shape structural guide
            1.0 * np.ones(len(X_my_tr))
        ])
        
        # 5. Train Joint LightGBM
        joint_model = LGBMRegressor(
            n_estimators=1200,          # Even more trees
            learning_rate=0.015,        # Even slower learning rate
            num_leaves=31,
            min_child_samples=10,
            reg_alpha=0.5,
            reg_lambda=0.5,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=RANDOM_SEED + fold, # Vary seed per fold to avoid artifacts
            n_jobs=-1
        )
        
        joint_model.fit(
            X_joint, 
            y_joint,
            sample_weight=w_joint
        )
        
        # 6. Predict on fold test set
        pred_joint_log = joint_model.predict(X_my_te)
        pred_joint = np.expm1(np.clip(pred_joint_log, -20, 20))
        
        rmse_fold = math.sqrt(mean_squared_error(y_my_te_raw, pred_joint))
        sp_fold = spearman_corr(y_my_te_raw, pred_joint)
        
        cv_rmse_joint.append(rmse_fold)
        cv_sp_joint.append(sp_fold)
        
        print(f'  Fold {fold+1}/{n_splits_my} -> RMSE: {rmse_fold:.4f} | Spearman: {sp_fold:.4f}')

    print(f'\n  Joint CV Average   -> RMSE: {np.mean(cv_rmse_joint):.4f} +/- {np.std(cv_rmse_joint):.4f}')
    print(f'  Joint CV Average   -> Spearman: {np.mean(cv_sp_joint):.4f} +/- {np.std(cv_sp_joint):.4f}')


MySQL fine-tune : 1091 rows / 37 workloads
MySQL held-out  : 223 rows / 9 workloads
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1

Pseudo-label Spearman: no valid workloads (need >=2 samples per workload).
Skipping pseudo-label gating and Stage 1 warm-up; doing Stage 2 (real-label refinement) only.

PG importance weights  mean: 0.000  std: 0.000

-- Stage 1: pseudo-label warm-up --
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1


/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/numpy/lib/function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/numpy/lib/function_base.py:2855: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it 

[LightGBM] [Info] Calculating query weights...
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.384704 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 12364
[LightGBM] [Info] Number of data points in the train set: 20758, number of used features: 155
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[Ligh

/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.108079 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1313
[LightGBM] [Info] Number of data points in the train set: 1091, number of used features: 75
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posi

...

/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006573 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 13662
[LightGBM] [Info] Number of data points in the train set: 21837, number of used features: 207
[LightGBM] [Info] Start training from score 2.686046
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gai

/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.461404 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 13681
[LightGBM] [Info] Number of data points in the train set: 21837, number of used features: 207
[LightGBM] [Info] Start training from score 2.378241
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

/home/E2ETune-AI4DB/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
